# 🏠 MietCheck · 🤖 QUA³CK-Phase 3: Algorithmenauswahl

**Modul:** Data Analytics & Big Data (IU, 4. Semester) · **Projekt:** MietCheck – „Zahle ich zu viel Miete?"

> Teil 3 von 6 der QUA³CK-Notebook-Serie. Reihenfolge:
> **Q**ualitätsprüfung → **U**nderstanding → **A**lgorithmenauswahl →
> Modellentwicklung → **C**ross-Validation → **K**nowledge.

**Ziel:** geeignete ML-Algorithmen für unser Regressionsproblem (Kaltmiete vorhersagen) auswählen und in einem schnellen Vergleich testen.

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import config as C
sns.set_theme(style='whitegrid', palette='rocket')
print('Setup ok ·', ROOT.name)

Setup ok · MietCheck


## 1. Kandidaten & Begründung

| Algorithmus | Warum? |
|---|---|
| **Lineare Regression** | einfache, interpretierbare Baseline |
| **Random Forest** | erfasst nichtlineare Effekte & Interaktionen, robust |
| **Gradient Boosting** | meist Top-Leistung auf tabellarischen Daten |

Alle drei stammen aus der Vorlesung (Kap. 3–5).

## 2. Datenaufbereitung als Pipeline
Imputation → Skalierung → One-Hot-Encoding (identisch zu `src/train.py`).

In [2]:
from train import build_preprocessor, get_models
pre = build_preprocessor(); pre

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('impute',
                                                  SimpleImputer(strategy='median')),
                                                 ('scale', StandardScaler())]),
                                 ['livingSpace', 'noRooms', 'yearConstructed']),
                                ('cat',
                                 Pipeline(steps=[('impute',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                min_frequency=30,
                                                                sparse_output=False))]),
                                 ['regio1', 'regio2', 'condition',
                                  'interiorQual', 'typeOfFlat']),
                                ('bool', 'passthrough',
                                 ['balcony', 'hasKitchen', 'cellar', 'lift',
                                  'garden', 'newlyConst'])])

## 3. Schneller Vergleich (Holdout, Stichprobe)
Zur Übersicht auf einer 40k-Stichprobe – die faire Auswahl per Kreuzvalidierung folgt in Phase C.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
df = pd.read_parquet(C.CLEAN_PARQUET).sample(40000, random_state=C.RANDOM_STATE)
X, y = df[C.FEATURES], df[C.TARGET]
Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=0.2,random_state=C.RANDOM_STATE)
res = {}
for name, pipe in get_models(build_preprocessor()).items():
    pipe.fit(Xtr,ytr); p = pipe.predict(Xte)
    res[name] = {'MAE': mean_absolute_error(yte,p), 'R2': r2_score(yte,p)}
    print(f'  {name:22} MAE {res[name]["MAE"]:6.1f} € · R² {res[name]["R2"]:.3f}')
res = pd.DataFrame(res).T

  Lineare Regression     MAE  124.7 € · R² 0.815


  Random Forest          MAE  110.3 € · R² 0.837


  Gradient Boosting      MAE   97.1 € · R² 0.876


In [4]:
ax = res['MAE'].sort_values().plot.barh(figsize=(8,3), color='#10b981')
ax.set(title='Holdout-MAE je Algorithmus (Stichprobe)', xlabel='MAE (€)')
plt.tight_layout(); plt.show()

## ✅ Fazit Phase A
- **Gradient Boosting** liefert schon in der Stichprobe den kleinsten Fehler.
- Bäume schlagen die lineare Baseline deutlich → nichtlineare Effekte sind wichtig.
- Endgültige Auswahl per 5-facher Kreuzvalidierung in **Phase C**.

➡️ Weiter mit **04 · Modellentwicklung**.